# 02 - Pré-processamento e Limpeza de Dados Vetorizada (NumPy & Pandas)
**Unidades Curriculares:** Aprendizagem Automática (AAUTIA) & Sistemas de Decisão e Planeamento (PLNTDIA)
**Dataset:** Waymo Open Motion Dataset

Este notebook demonstra a limpeza e filtragem física dos dados brutos com recurso exclusivo a **NumPy** e **Pandas**, garantindo:
1. Eliminação de ruído de sensores, valores estáticos e medições fisicamente impossíveis.
2. Contratos estritos de separação de colunas (Metadados, Coordenadas Globais, Features de Entrada e Targets).
3. Partição de dados por cenários (`scenario_id`) com **zero fuga de dados** (*no data leakage*).


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Adicionar raiz do projeto ao path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.features.pandas_feature_pipeline import PandasFeaturePipeline
print("Pipeline importado com sucesso!")


## 1. Carregamento do Dataset Extraído


In [ ]:
data_path = ROOT / "data" / "processed" / "predict_to_plan_samples.csv"
df_raw = pd.read_csv(data_path)
print(f"Total de amostras carregadas: {len(df_raw)}")
print(f"Colunas presentes ({len(df_raw.columns)}):", list(df_raw.columns[:10]), "...")
df_raw.head(3)


## 2. Auditoria e Limpeza Vetorizada de Valores Nulos e Outliers Físicos


In [ ]:
# Limpeza e filtragem física
df_clean = PandasFeaturePipeline.clean_and_filter_outliers(df_raw)
print(f"Amostras antes da limpeza: {len(df_raw)}")
print(f"Amostras após filtragem física: {len(df_clean)} (Retenção: {len(df_clean)/len(df_raw)*100:.2f}%)")


## 3. Divisão em Conjuntos de Treino, Validação e Teste por Cenários (Group Split)
Para garantir que medições contínuas de um mesmo episódio temporal de 20 segundos não contaminem o teste, a divisão é efetuada estritamente ao nível do `scenario_id`.


In [ ]:
unique_scenarios = pd.Series(df_clean["scenario_id"].unique())
shuffled_scenarios = unique_scenarios.sample(frac=1.0, random_state=42).tolist()

n_scen = len(shuffled_scenarios)
train_end = int(n_scen * 0.70)
val_end = int(n_scen * 0.85)

train_scens = set(shuffled_scenarios[:train_end])
val_scens = set(shuffled_scenarios[train_end:val_end])
test_scens = set(shuffled_scenarios[val_end:])

df_train = df_clean[df_clean["scenario_id"].isin(train_scens)].reset_index(drop=True)
df_val = df_clean[df_clean["scenario_id"].isin(val_scens)].reset_index(drop=True)
df_test = df_clean[df_clean["scenario_id"].isin(test_scens)].reset_index(drop=True)

print(f"Partição concluída:")
print(f"  • Treino:     {len(df_train)} amostras ({len(train_scens)} cenários)")
print(f"  • Validação:  {len(df_val)} amostras ({len(val_scens)} cenários)")
print(f"  • Teste:      {len(df_test)} amostras ({len(test_scens)} cenários)")
print("Verificação de Sobreposição (Data Leakage):", 
      len(train_scens.intersection(val_scens)) == 0 and len(train_scens.intersection(test_scens)) == 0)


## 4. Estatísticas Descritivas (Pandas Aggregation)


In [ ]:
stats = PandasFeaturePipeline.compute_summary_statistics(df_clean)
stats
